# 1.2 BPE 训练

字节级分词没有「未登录词」问题，但序列会比较长。BPE（byte-pair encoding）通过反复合并高频的相邻 token pair，
从 256 个基础字节出发构建更大的词表，用更短的序列保留字节级分词的灵活性。本节先实现完整训练流程；优化实现见 scripts/train_bpe.py。

In [ ]:
import os
import sys

# 直接指定项目根目录，后面的路径一律相对它来写；需替换成自己实际的项目根目录的绝对路径
os.chdir(r"C:\Users\intangible\Desktop\easy-llm")

# 注意切目录并不会让 import 找到 scripts/ 下的模块，还得把它加进模块搜索路径
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm


In [ ]:
# 纯字节分词会把序列拉得很长；
# 一句 9 个词的英文，按「单词」看只有 9 个 token，按「字节」看却要 41 个。
sentence = "Once upon a time, there was a little girl"
words = sentence.split()
print("单词数:", len(words))
print("字节数:", len(sentence.encode("utf-8")))

# 而单纯的词级别分词（一个词一个 token）会让词表长度爆炸，那我们就可以取两者折中，即子词。
# 那些常见的固定短语，词组等就该被视作为一个子词（统计单元/语义单元），而不是零散的多个字符
# 中文示例：人工智能，北京大学，端午节

In [ ]:
# 预分词：先把文本切成若干片段，之后 BPE 只在片段内部统计与合并。
# PAT 的匹配顺序是英文缩写、字母、数字、标点/符号和空白。词或标点前的一个空格会并入当前片段。
# 其中 \p{L}+ 匹配字母，\p{N}+ 匹配数字，[^...] 匹配标点/符号，\s+ 匹配空白。
# 因此 "i'll" 会拆成 "i" 与 "'ll"，数字、标点和空白也会按规则分别保留。

import regex as re
# PAT 是 GPT‑2 原生使用的预分词正则表达式（主要面向英文）
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""" 

# 按 PAT 正则规则，从左往右提取所有匹配到的预分词片段
print(re.findall(PAT, "i'll buy 3 apples; price: $99.99!")) 
print(re.findall(PAT, "123+256"))
print(re.findall(PAT, "111 111 111 111")) # GPT-2 版本下大模型看到的是一个超长 token，想要学明白这个 token 到底包含多少个 1 还是太难了；
# 后来，GPT-4 对 PAT 进行优化，每段的连续数字不超过 3 个，单个 token 更短，方便模型学习

In [ ]:
# 先按特殊 token 切分，再对普通片段应用 PAT，返回「片段 -> 出现次数」的计数表。

print(re.findall(PAT, "<|endoftext|>")) # 顺序不能乱，否则特殊 token 可能会被拆散，无法被正确识别

chunk_text = "The mole felt so safe with the little girl. <|endoftext|>Once upon a time,"
print(re.split(re.escape("<|endoftext|>"), chunk_text)) # re.escape()：把字符串中所有正则特殊符号全部转义，按字面文本分割

def preprocess_chunks(chunk_text: str, special_tokens_pattern: str) -> dict[str, int]:
    '''
        对文本块做 BPE 训练前预处理，统计预分词片段的出现频次。
        chunk_text: 原始输入文本块，可能包含 <|endoftext|> 等特殊 token
        special_tokens_pattern: 正则模式字符串，由 re.escape 转义后的全部特殊 token 拼接而成；特殊 token 本身不会保留到输出统计字典。
    '''
    chunk_dict = {}
    # 第一步：按特殊 token 边界切分；特殊 token 不进入普通片段。
    for chunk1 in re.split(special_tokens_pattern, chunk_text):
        # 第二步：对每个普通片段应用 PAT，并统计片段频次。
        for chunk2 in re.findall(PAT, chunk1):
            chunk_dict[chunk2] = chunk_dict.get(chunk2, 0) + 1 # .get 方法安全获取 chunk_dict[chunk2] 值，不存在则返回 0 
    return chunk_dict # {low:5, lower:2, widest:3, newest:6}

In [ ]:
# BPE 训练：每轮完整统计 pair，再完整遍历所有片段执行合并。

def train_bpe_reference(input_path, vocab_size, special_tokens):
    """
    input_path: str  训练语料文件路径
    vocab_size: int  目标词表大小（包含基础 256 字节 + 特殊 token + 合并生成的子词）
    special_tokens: list[str]  特殊 token 列表，例如 ["<|endoftext|>"]
    return: (vocab, merges)
        vocab: dict[int, bytes]  token ID 到字节串 (bytes) 的映射；key 为 token_id，value 为对应的字节序列
        merges: list[tuple[bytes, bytes]]  按创建顺序排列的合并规则，每一项是(左片段字节, 右片段字节)
    """

    # 1. 初始化词表
    # vocab：词表，token_id → bytes。先初始化 0‑255，对应 256 个基础 UTF‑8 字节。
    vocab = {i: bytes([i]) for i in range(256)}
    # 将特殊 token 编码为字节序列，接续 256 之后的 ID 存入词表
    for i, st in enumerate(special_tokens):
        vocab[i + 256] = st.encode("utf-8")

    merges = []  # 存储 BPE 合并规则
    tokens = {}  # 字符串片段（token id 形式）的出现情况；key：id 序列，比如 "hello" -> (32, 104, 101, 108, 108, 111)）；value：该片段出现的总频次 


    # 2. 预分词 = 特殊 token 分割 + GPT-2 正则分词规则
    # 构造特殊 token 分割正则：每个特殊 token 先 re.escape 转义，再用正则 |（或）拼接
    # 无特殊 token 时使用 r"(?!)" 永远不匹配，re.split 不会做任何切分
    special_pattern = "|".join(re.escape(st) for st in special_tokens) if special_tokens else r"(?!)" 

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:  # 遍历 txt 文件中的每一行文本，一行文本对应一个完整句子/完整段落
            line = line.rstrip("\n")  # 移除行尾换行符
            
            for chunk, count in preprocess_chunks(line, special_pattern).items(): # 执行预分词，返回 {预分词字符串片段: 出现次数}，再进行遍历 
                ids = tuple(chunk.encode("utf-8")) # 将预分词得到的字符串片段先编码为 utf‑8 字节，再转成 id 序列（BPE 是跑在 token id 层面的）；元组可作为字典 key，列表不可以
                tokens[ids] = tokens.get(ids, 0) + count # 更新当前 id 序列的出现频次

    # 3. 循环执行 BPE 合并，直到词表大小达到目标 vocab_size
    while len(vocab) < vocab_size:
        # 1) 统计每个片段中相邻 token pair 的共现频次，并汇总到一个字典里
        stats = {}  # key: 一对相邻 token_id (p0, p1)；value：这对 pair 在全部片段中的出现次数

        for ids, count in tokens.items(): # 独立统计每个预分词片段中 token pair 的出现频次 
            # ids: (108, 111, 119, 101)  ids[1:]: (111, 119, 101)
            # count: 2
            for p0, p1 in zip(ids, ids[1:]): # 遍历序列中每一组相邻的 token 对 (ids[i], ids[i+1])
                stats[(p0, p1)] = stats.get((p0, p1), 0) + count # 更新 (p0, p1) 的出现频次
        
        if not stats:  # 不存在任何可合并的相邻 pair，提前终止循环
            break

        # 2) 选出频次最高的 pair，生成新 token 加入词表
        # max 取最大 pair；key：优先按共现频次，频次相同时按字典序比较字节串（abd > abc），保证平局时的处理是确定的
        p0, p1 = max(stats, key=lambda k: (stats[k], vocab[k[0]], vocab[k[1]]))
        new_id = len(vocab)  # 新 token 的 ID 等于当前词表长度，ID 自增，与 merges 顺序一一对应
        vocab[new_id] = vocab[p0] + vocab[p1]  # 新 token 字节 = 左片段字节与右片段字节的拼接（老 token 的字节肯定求出来了）
        merges.append((vocab[p0], vocab[p1]))  # 保存本次合并规则

        # 3) 完整遍历全部 tokens，执行本轮 pair 合并；对每一个 id 序列从左向右扫描，将 (p0,p1) 替换为 new_id
        new_tokens = {}  # 保存合并之后新的 id 序列以及对应频次
        for ids, count in tokens.items():
            # (p0, p1): (119, 101)
            # ids: (108, 111, 119, 101) 
            # merged: [108, 111, new_id]
            # count: 2
            merged = []
            i = 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == p0 and ids[i + 1] == p1: # 检测当前位与下一位是否构成待合并的 (p0,p1)，匹配则替换成 new_id
                    merged.append(new_id) 
                    i += 2
                else: # 不匹配则保留当前 ID
                    merged.append(ids[i])
                    i += 1  

            merged = tuple(merged)  # 转为元组才能当字典 key
            new_tokens[merged] = new_tokens.get(merged, 0) + count # 更新 merged 对应片段的出现频次

        tokens = new_tokens  # 将 tokens 更新为合并后的 new_tokens

    # 实际 BPE 训练脚本还需要：把 vocab 和 merges 保存在磁盘上

    return vocab, merges


In [ ]:
# max(..., key=lambda ...)：`key` 必须传入一个函数；这个函数接收迭代出来的每一个元素，返回「用于比较大小的值」。
stats = {(b"s", b"t"): 9, (b"e", b"s"): 9, (b"l", b"o"): 5}
best = max(stats, key=lambda pair: (stats[pair], pair[0], pair[1])) # 先按 pair 频次选择，频次相同再按字节序处理平局。

# 普通函数写法
def get_compare_key(pair): 
    return (stats[pair], pair[0], pair[1])

best = max(stats, key=get_compare_key)

print(best, stats[best])
# -> (b's', b't') 9

In [ ]:
# 在一个极小的临时语料上跑通参考实现（词表上限 300，仅作验证）。

import tempfile

tmp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
tmp.write("low low low low low\nlower lower widest widest widest\nnewest newest newest newest newest newest\n")
tmp.close()

vocab, merges = train_bpe_reference(tmp.name, 300, ["<|endoftext|>"])
print("词表大小:", len(vocab))
print("前 6 条 merges:", [a.decode() + b.decode() for a, b in merges[:6]])
os.unlink(tmp.name)
# -> 词表大小: 273
# -> 前 6 条 merges: ['st', 'est', 'ow', 'low', 'west', 'ne']（与官方例子一致）